# Projet NLP : Named entity recognition du titulaire / suppléant et de son affiliation politique

# Objectifs

On souhaite évaluer la performance d'une méthode Named-Entity-Recognition pour l'identification du titulaire de la profession de foi et de son affiliation politique. Pour cela, on tire parti des annotations disponibles, pour tagger les noms des titulaires / suppléants dans les transcriptions.

Plus précisément, on évalue ici la performance de GlinNER2 car : 
- c'est un modèle de type transformer de petite taille, uniquement CPU
- le modèle permet de faire du zero-shot learning et permet d'indiquer que l'on souhaite avoir l'affiliation politique correspondante à l'entité reconnue

# Annotations / création d'un jeu de test

On annote les professions de foi à partir des métadonnées disponibles.
Résulat : 
- liste des segments de texte où apparaissent le nom du titulaire / suppléant 
- affiliation politique de la profession de foi

## Lecture des métadonnées

In [11]:
import contextlib
import pandas as pd
import polars as pl

In [7]:
csv = pd.read_csv("archelect_search.csv")
csv['date'] = pd.to_datetime(csv['date'])

/tmp/ipykernel_2178/2974932302.py:1: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  csv = pd.read_csv("archelect_search.csv")


In [8]:
# select 1988 election (because we only have transcriptions for this election)
csv1988 = csv.loc[(csv['date'].dt.year==1988) & (csv['contexte-election']=='législatives')]

In [9]:
for row in csv1988.itertuples():
    with contextlib.suppress(FileNotFoundError):
        with open("text_files/1988/legislatives/"+row.id+".txt", "r") as file:
            content = file.read()
        csv1988.loc[row.Index,"transcript"] = content
        csv1988.loc[row.Index,"transcript_length"] = len(content)

## Normalisation du nom

In [10]:
# On utilise la librairie unidecode pour normaliser le nom du titulaire

from unidecode import unidecode
s = "stävänger"

# Convert characters to their closest ASCII equivalents
res = unidecode(s)
print(res)

stavanger


In [28]:
df = pl.from_pandas(csv1988)

In [35]:
df = df.with_columns(
    transcript_normalized = pl.col("transcript").str.to_lowercase()
      .str.normalize("NFKD")
      .str.replace_all(r"\p{CombiningMark}", ""),
    titulaire_nom_normalized = pl.col("titulaire-nom").str.to_lowercase()
      .str.normalize("NFKD")
      .str.replace_all(r"\p{CombiningMark}", ""),
    titulaire_nom_is_in_transcript = pl.col("transcript_normalized")
        .str.contains(pl.col("titulaire_nom_normalized"))

)

In [36]:
df.select(pl.selectors.by_name("titulaire-nom","titulaire_nom_normalized","titulaire_nom_is_in_transcript"))

titulaire-nom,titulaire_nom_normalized,titulaire_nom_is_in_transcript
str,str,bool
"""Aulagne""","""aulagne""",true
"""Pujol""","""pujol""",true
"""Boyon""","""boyon""",true
"""Saint-Pierre""","""saint-pierre""",true
"""Mornet""","""mornet""",true
…,…,…
"""Hoarau""","""hoarau""",true
"""Pihouee""","""pihouee""",true
"""Hoarau""","""hoarau""",true


In [37]:
df.get_column("titulaire_nom_is_in_transcript").value_counts()

titulaire_nom_is_in_transcript,count
bool,u32
false,129
null,1
true,3410


## Position du nom dans le transcript

In [39]:
df = df.with_columns(
    position_nom = pl.col("transcript_normalized").str.find(pl.col("titulaire_nom_normalized"))
)

In [42]:
df.select(pl.selectors.by_name("transcript","titulaire-nom","position_nom","titulaire_nom_is_in_transcript"))

transcript,titulaire-nom,position_nom,titulaire_nom_is_in_transcript
str,str,u32,bool
"""Élections Législatives du 5 Ju…","""Aulagne""",1240,true
"""RÉPUBLIQUE FRANÇAISE - Liberté…","""Pujol""",208,true
"""Sciences Po / fonds CEVIPOF EL…","""Boyon""",155,true
"""Sciences Po / fonds CEVIPOF RÉ…","""Saint-Pierre""",102,true
"""RÉPUBLIQUE FRANÇAISE - LIBERTÉ…","""Mornet""",84,true
…,…,…,…
"""Claude HOARAU Égalité Responsa…","""Hoarau""",7,true
"""Sciences Po / fonds CEVIPOF An…","""Pihouee""",2017,true
"""Élie HOARAU Égalité Responsabi…","""Hoarau""",5,true


In [44]:
df.filter(pl.col("position_nom").is_null())

id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,identifiant de circonscription,images,pdf,ocr_url,titulaire-nom,titulaire-prenom,titulaire-sexe,titulaire-age,titulaire-age-calcule,titulaire-age-tranche,titulaire-profession,titulaire-mandat-en-cours,titulaire-mandat-passe,titulaire-associations,titulaire-autres-statuts,titulaire-soutien,titulaire-liste,titulaire-decorations,suppleant-nom,suppleant-prenom,suppleant-sexe,suppleant-age,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations,transcript,transcript_length,transcript_normalized,titulaire_nom_normalized,titulaire_nom_is_in_transcript,position_nom
str,datetime[μs],str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,bool,u32
"""EL174_L_1988_06_005_01_1_PF_04""",1988-06-05 00:00:00,"""Ve République;Assemblée Nation…","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""05""","""Hautes-Alpes""","""05 - Hautes-Alpes""","""1""","""https://ia803408.us.archive.or…","""https://ia803408.us.archive.or…","""https://ia803408.us.archive.or…","""Chevalier""","""Daniel""","""homme""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""Majorité présidentielle pour l…","""non""","""Graglia""","""Georges""","""non déterminé""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""Majorité présidentielle pour l…","""non""","""ELECTIONS LEGISLATIVES DU 5 JU…",1068.0,"""elections legislatives du 5 ju…","""chevalier""",false,null
"""EL174_L_1988_06_006_09_1_PF_02""",1988-06-05 00:00:00,"""Ve République;Assemblée Nation…","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""06""","""Alpes-Maritimes""","""06 - Alpes-Maritimes""","""9""","""https://ia801804.us.archive.or…","""https://ia801804.us.archive.or…","""https://ia801804.us.archive.or…","""Vassalo""","""Georges""","""homme""","""non mentionné""","""non mentionné""","""non mentionné""","""principal collège""","""conseiller municipal""","""maire;conseiller régional""","""non mentionné""","""non mentionné""","""Parti communiste français""","""Rassemblement des forces de ga…","""non""","""Bernasconi""","""Pierre""","""homme""","""non mentionné""","""non mentionné""","""non mentionné""","""professeur""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""Parti communiste français""","""Rassemblement des forces de ga…","""non""","""DÉPARTEMENT DES ALPES-MARITIME…",3891.0,"""departement des alpes-maritime…","""vassalo""",false,null
"""EL174_L_1988_06_010_01_1_PF_06""",1988-06-05 00:00:00,"""Élections législatives;Ve Répu…","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""10""","""Aube""","""10 - Aube""","""1""","""https://ia803407.us.archive.or…","""https://ia803407.us.archive.or…","""https://ia803407.us.archive.or…","""non mentionné""","""non mentionné""","""non déterminé""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""Parti ouvrier européen""","""non mentionné""","""non""","""non mentionné""","""non mentionné""","""non déterminé""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""Parti ouvrier européen""","""non mentionné""","""non""","""PARTI OUVRIER EUROPÉEN Siège :…",5281.0,"""parti ouvrier europeen siege :…","""non mentionne""",f